# अध्याय 7: चैट एप्लिकेशन बनाना
## OpenAI API त्वरित शुरुआत

यह नोटबुक [Azure OpenAI सैंपल्स रिपॉजिटरी](https://github.com/Azure/azure-openai-samples?WT.mc_id=academic-105485-koreyst) से अनुकूलित है जिसमें नोटबुक शामिल हैं जो [Azure OpenAI](notebook-azure-openai.ipynb) सेवाओं तक पहुंचते हैं।

Python OpenAI API कुछ संशोधनों के साथ Azure OpenAI मॉडल के साथ भी काम करता है। यहां अधिक जानें कि इन दोनों के बीच क्या अंतर है: [Python के साथ OpenAI और Azure OpenAI एंडपॉइंट्स के बीच कैसे स्विच करें](https://learn.microsoft.com/azure/ai-foundry/openai/how-to/switching-endpoints?WT.mc_id=academic-109527-jasmineg)


# अवलोकन  
"बड़े भाषा मॉडल ऐसे फ़ंक्शन होते हैं जो टेक्स्ट को टेक्स्ट में मैप करते हैं। दिए गए इनपुट टेक्स्ट स्ट्रिंग के लिए, एक बड़ा भाषा मॉडल उस टेक्स्ट की भविष्यवाणी करने की कोशिश करता है जो अगला आने वाला है"(1)। यह "क्विकस्टार्ट" नोटबुक उपयोगकर्ताओं को उच्च-स्तरीय LLM अवधारणाओं, AML के साथ शुरू करने के लिए मुख्य पैकेज आवश्यकताओं, प्रॉम्प्ट डिजाइन का एक सौम्य परिचय, और विभिन्न उपयोग मामलों के कई छोटे उदाहरणों से परिचित कराएगा। 


## सामग्री तालिका  

[समीक्षा](#overview)  
[OpenAI सेवा का उपयोग कैसे करें](#how-to-use-openai-service)  
[1. अपनी OpenAI सेवा बनाना](#1.-creating-your-openai-service)  
[2. स्थापना](#2.-installation)    
[3. प्रमाणपत्र](#3.-credentials)  

[उपयोग के मामले](#use-cases)    
[1. पाठ का संक्षेप करें](#1.-summarize-text)  
[2. पाठ वर्गीकृत करें](#2.-classify-text)  
[3. नए उत्पाद नाम बनाएँ](#3.-generate-new-product-names)  
[4. एक वर्गीकारक को ठीक करें](#4.fine-tune-a-classifier)  

[संदर्भ](#references)


### अपना पहला प्रॉम्प्ट बनाएं  
यह छोटा अभ्यास "सारांश" नामक एक सरल कार्य के लिए OpenAI मॉडल को प्रॉम्प्ट सबमिट करने का मौलिक परिचय प्रदान करेगा।  


**चरण**:  
1. अपने पायथन पर्यावरण में OpenAI लाइब्रेरी स्थापित करें  
2. मानक सहायक लाइब्रेरी लोड करें और उस OpenAI सेवा के लिए अपनी सामान्य OpenAI सुरक्षा प्रमाण पत्र सेट करें जिसे आपने बनाया है  
3. अपने कार्य के लिए एक मॉडल चुनें  
4. मॉडल के लिए एक सरल प्रॉम्प्ट बनाएं  
5. अपने अनुरोध को मॉडल API को सबमिट करें!  


### 1. OpenAI स्थापित करें


In [ ]:
%pip install openai python-dotenv

### 2. सहायक पुस्तकालय आयात करें और क्रेडेंशियल्स बनाएं


In [ ]:
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

API_KEY = os.getenv("OPENAI_API_KEY","")
assert API_KEY, "ERROR: OpenAI Key is missing"

client = OpenAI(
    api_key=API_KEY
    )


### 3. सही मॉडल खोजना  
GPT-4o और GPT-4o मिनी जैसे मॉडल प्राकृतिक भाषा को समझ सकते हैं और उत्पन्न कर सकते हैं, और OpenAI प्लेटफ़ॉर्म पर विभिन्न कार्यों के लिए उपयुक्त विभिन्न शक्ति और गति के स्तरों के साथ उपलब्ध हैं।


In [ ]:
# Select a general purpose chat model
model = "gpt-5-mini"


## 4. प्रॉम्प्ट डिज़ाइन  

"बड़े भाषा मॉडल का जादू यह है कि विशाल मात्रा में टेक्स्ट पर इस भविष्यवाणी त्रुटि को कम करने के लिए प्रशिक्षित होने से, मॉडल उन अवधारणाओं को सीख लेते हैं जो इन भविष्यवाणियों के लिए उपयोगी होती हैं। उदाहरण के लिए, वे निम्न अवधारणाएँ सीखते हैं"(1):

* कैसे वर्तनी करें
* व्याकरण कैसे काम करता है
* कैसे पैराफ्रेज़ करें
* कैसे प्रश्नों का उत्तर दें
* कैसे वार्तालाप करें
* कई भाषाओं में कैसे लिखें
* कैसे कोड करें
* आदि।

#### बड़े भाषा मॉडल को नियंत्रित कैसे करें  
"बड़े भाषा मॉडल में सभी इनपुट में से, सबसे प्रभावशाली स्पष्ट रूप से टेक्स्ट प्रॉम्प्ट होता है(1)।

बड़े भाषा मॉडल को आउटपुट उत्पन्न करने के लिए कुछ तरीकों से प्रॉम्प्ट किया जा सकता है:

निर्देश: मॉडल को बताएं कि आप क्या चाहते हैं
पूर्णता: मॉडल को आपके चाहने की शुरुआत को पूरा करने के लिए प्रेरित करें
प्रदर्शन: मॉडल को दिखाएं कि आप क्या चाहते हैं, या तो:
प्रॉम्प्ट में कुछ उदाहरण
फाइन-ट्यूनिंग प्रशिक्षण डेटासेट में सैंकड़ों या हजारों उदाहरण"



#### प्रॉम्प्ट बनाने के तीन मूलभूत दिशानिर्देश हैं:

**दिखाएँ और बताएं**। स्पष्ट करें कि आप क्या चाहते हैं, निर्देशों, उदाहरणों या दोनों के संयोजन के माध्यम से। यदि आप चाहते हैं कि मॉडल आइटम की सूची को वर्णानुक्रम में क्रमबद्ध करे या पैराग्राफ़ को भावना के अनुसार वर्गीकृत करे, तो इसे दिखाएँ कि आप वही चाह रहे हैं।

**गुणवत्ता डेटा प्रदान करें**। यदि आप कोई क्लासिफायर बनाना चाहते हैं या मॉडल से कोई पैटर्न फॉलो करवाना चाहते हैं, तो यह सुनिश्चित करें कि पर्याप्त उदाहरण हों। अपने उदाहरणों को सावधानी से प्रूफरीड करें — मॉडल आमतौर पर बुनियादी वर्तनी की गलतियों को देखकर प्रतिक्रिया दे देता है, लेकिन यह भी मान सकता है कि यह इरादतन है और इससे प्रतिक्रिया प्रभावित हो सकती है।

**अपनी सेटिंग्स चेक करें।** तापन (temperature) और top_p सेटिंग्स नियंत्रित करती हैं कि मॉडल प्रतिक्रिया उत्पन्न करने में कितना निर्धारक है। यदि आप ऐसे जवाब पूछ रहे हैं जहाँ केवल एक सही उत्तर है, तो इन सेटिंग्स को कम रखना बेहतर होगा। यदि आप अधिक विविध प्रतिक्रियाएँ चाहते हैं, तो इन्हें उच्च रखना उचित होगा। लोग सबसे बड़ी गलती यह मानकर करते हैं कि ये सेटिंग्स "बुद्धिमत्ता" या "रचनात्मकता" नियंत्रण हैं।


स्रोत: https://learn.microsoft.com/azure/ai-foundry/openai/overview


### 5. प्रस्तुत करें!


In [ ]:
# Create your first prompt
text_prompt = "Should oxford commas always be used?"

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},],
  store=False,)

response.output_text


### एक ही कॉल को दोहराएं, परिणामों की तुलना कैसे होती है?


In [ ]:

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},],
  store=False,)

response.output_text


## पाठ संक्षेप करें  
#### चुनौती  
एक पाठ खंड के अंत में 'tl;dr:' जोड़कर पाठ संक्षेप करें। ध्यान दें कि मॉडल बिना अतिरिक्त निर्देशों के कई कार्य कैसे करता है। आप मॉडल के व्यवहार को संशोधित करने और प्राप्त संक्षेपण को अनुकूलित करने के लिए tl;dr की तुलना में अधिक वर्णनात्मक संकेतों के साथ प्रयोग कर सकते हैं(3)।  

हालिया कार्यों ने एक बड़े पाठ संकलन पर पूर्व-प्रशिक्षण और उसके बाद किसी विशिष्ट कार्य पर फाइन-ट्यूनिंग करके कई NLP कार्यों और बेंचमार्क पर महत्वपूर्ण उन्नति दिखाई है। हालांकि वास्तुकला में आमतौर पर टास्क-ऐग्नोस्टिक होती है, इस विधि में अभी भी हजारों या दसियों हजारों उदाहरणों के टास्क-विशिष्ट फाइन-ट्यूनिंग डेटासेट की आवश्यकता होती है। इसके विपरीत, मनुष्य आमतौर पर कुछ उदाहरणों या सरल निर्देशों से नया भाषा कार्य कर सकते हैं - जो वर्तमान NLP सिस्टम अभी भी काफी हद तक करने में संघर्ष कर रहे हैं। यहां हम दिखाते हैं कि भाषा मॉडलों का स्केलिंग कार्य-ऐग्नोस्टिक, फ्यू-शॉट प्रदर्शन को काफी सुधारता है, कभी-कभी पिछले राज्य-कल के फाइन-ट्यूनिंग तरीकों के साथ प्रतिस्पर्धात्मक भी होता है।  



संक्षेप  


# कई उपयोग मामलों के लिए अभ्यास  
1. टेक्स्ट सारांशित करें  
2. टेक्स्ट वर्गीकृत करें  
3. नए उत्पाद नाम उत्पन्न करें


In [ ]:
prompt = "Recent work has demonstrated substantial gains on many NLP tasks and benchmarks by pre-training on a large corpus of text followed by fine-tuning on a specific task. While typically task-agnostic in architecture, this method still requires task-specific fine-tuning datasets of thousands or tens of thousands of examples. By contrast, humans can generally perform a new language task from only a few examples or from simple instructions - something that current NLP systems still largely struggle to do. Here we show that scaling up language models greatly improves task-agnostic, few-shot performance, sometimes even reaching competitiveness with prior state-of-the-art fine-tuning approaches.\n\nTl;dr"


In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},],
  store=False,)

response.output_text


## पाठ वर्गीकरण करें  
#### चुनौती  
आइटमों को उन श्रेणियों में वर्गीकृत करें जो पूर्वानुमान के समय प्रदान की गई हैं। निम्न उदाहरण में, हम श्रेणियों और वर्गीकृत करने के लिए पाठ दोनों को संकेत में प्रदान करते हैं (*playground_reference)। 

ग्राहक पूछताछ: नमस्ते, मेरी लैपटॉप कीबोर्ड की एक कुंजी हाल ही में टूट गई है और मुझे उसका प्रतिस्थापन चाहिए:

वर्गीकृत श्रेणी:


In [ ]:
prompt = "Classify the following inquiry into one of the following: categories: [Pricing, Hardware Support, Software Support]\n\ninquiry: Hello, one of the keys on my laptop keyboard broke recently and I'll need a replacement:\n\nClassified category:"
print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},],
  store=False,)

response.output_text


## नए उत्पाद नाम उत्पन्न करें
#### चुनौती
उदाहरण शब्दों से उत्पाद नाम बनाएं। यहां हम प्रॉम्प्ट में उस उत्पाद के बारे में जानकारी शामिल करते हैं जिसके लिए हम नाम उत्पन्न करने जा रहे हैं। हम एक समान उदाहरण भी प्रदान करते हैं ताकि उस पैटर्न को दिखाया जा सके जिसे हम प्राप्त करना चाहते हैं। हमने अधिक यादृच्छिकता और अधिक अभिनव प्रतिक्रियाओं के लिए तापमान मान भी उच्च सेट किया है।

उत्पाद विवरण: एक होम मिल्कशेक मेकर
बीज शब्द: तेज, स्वस्थ, कॉम्पैक्ट।
उत्पाद नाम: होमशेकर, फिट शेकर, क्विकशेक, शेक मेकर

उत्पाद विवरण: एक जोड़ी जूते जो किसी भी पैर के आकार में फिट हो सकते हैं।
बीज शब्द: अनुकूलनीय, फिट, ऑम्नी-फिट।


In [ ]:
prompt = "Product description: A home milkshake maker\nSeed words: fast, healthy, compact.\nProduct names: HomeShaker, Fit Shaker, QuickShake, Shake Maker\n\nProduct description: A pair of shoes that can fit any foot size.\nSeed words: adaptable, fit, omni-fit."

print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt}],
  store=False,)

response.output_text


# संदर्भ  
- [Openai Cookbook](https://github.com/openai/openai-cookbook?WT.mc_id=academic-105485-koreyst)  
- [Microsoft Foundry portal](https://ai.azure.com?WT.mc_id=academic-105485-koreyst)  
- [टेक्स्ट वर्गीकरण के लिए GPT मॉडलों को फाइन-ट्यून करने के सर्वोत्तम अभ्यास](https://platform.openai.com/docs/guides/fine-tuning?WT.mc_id=academic-105485-koreyst)


# अधिक सहायता के लिए  
[OpenAI Commercialization Team](AzureOpenAITeam@microsoft.com) 


# योगदानकर्ता
* Louis Li  


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:
इस दस्तावेज़ का अनुवाद AI अनुवाद सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) का उपयोग करके किया गया है। जबकि हम सटीकता के लिए प्रयास करते हैं, कृपया ध्यान दें कि स्वचालित अनुवादों में त्रुटियाँ या अशुद्धियाँ हो सकती हैं। मूल दस्तावेज़ अपनी मूल भाषा में ही प्रामाणिक स्रोत माना जाना चाहिए। महत्वपूर्ण जानकारी के लिए, पेशेवर मानव अनुवाद की सिफारिश की जाती है। इस अनुवाद के उपयोग से उत्पन्न किसी भी गलतफहमी या गलत व्याख्या के लिए हम उत्तरदायी नहीं हैं।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
